<a href="https://colab.research.google.com/github/kirti73/AI_lab/blob/main/WEEK10/1BF24CS140_WEEK10_APLHABETA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Alpha-Beta Pruning Implementation

We will define a Tree structure where each node can have children (for internal nodes) or a value (for leaf nodes). The algorithm will perform a depth-first search, maintaining and updating `alpha` (the best value MAX can secure) and `beta` (the best value MIN can secure) to prune unnecessary branches.

In [ ]:
class Node:
    def __init__(self, name, value=None, children=None):
        self.name = name
        self.value = value
        self.children = children if children is not None else []

# Helper function to print tree structure
def print_tree(node, level=0):
    indent = "  " * level
    val_str = f" (val={node.value})" if node.value is not None else ""
    print(f"{indent}{node.name}{val_str}")
    for child in node.children:
        print_tree(child, level + 1)

### Alpha-Beta Search Algorithm

This function executes the search, records the chosen path, and identifies pruned nodes.

In [ ]:
pruned_branches = []
path_taken = []

def alpha_beta(node, depth, alpha, beta, is_max, path):
    # Base case: Leaf node
    if not node.children or node.value is not None:
        return node.value, path + [node.name]

    current_path = path + [node.name]

    if is_max:
        max_eval = float('-inf')
        best_path = []
        for child in node.children:
            eval_val, child_path = alpha_beta(child, depth + 1, alpha, beta, False, current_path)
            if eval_val > max_eval:
                max_eval = eval_val
                best_path = child_path
            alpha = max(alpha, eval_val)
            if beta <= alpha:
                # Record pruned siblings
                child_idx = node.children.index(child)
                for pruned_child in node.children[child_idx + 1:]:
                    pruned_branches.append((node.name, pruned_child.name))
                break  # Beta cutoff
        return max_eval, best_path
    else:
        min_eval = float('inf')
        best_path = []
        for child in node.children:
            eval_val, child_path = alpha_beta(child, depth + 1, alpha, beta, True, current_path)
            if eval_val < min_eval:
                min_eval = eval_val
                best_path = child_path
            beta = min(beta, eval_val)
            if beta <= alpha:
                # Record pruned siblings
                child_idx = node.children.index(child)
                for pruned_child in node.children[child_idx + 1:]:
                    pruned_branches.append((node.name, pruned_child.name))
                break  # Alpha cutoff
        return min_eval, best_path

### Define and Run on a Sample Tree

Let's construct a typical 3-level tree to demonstrate our solver.

In [13]:
def build_tree_from_input():
    print("--- Build Your Custom Tree ---")
    print("We will construct a tree where internal nodes branch, and leaf nodes hold numeric values.\n")

    nodes = {}

    # Helper to get or create a node
    def get_or_create(name):
        if name not in nodes:
            nodes[name] = Node(name)
        return nodes[name]

    # Get leaf values first
    try:
        num_leaves = int(input("Enter the number of leaf nodes: "))
        for _ in range(num_leaves):
            name = input("Enter leaf node name: ").strip()
            val = float(input(f"Enter evaluation value for leaf '{name}': "))
            nodes[name] = Node(name, value=val)

        # Connect parents to children
        print("\nNow define internal connections. Enter parent and its children (comma-separated).")
        print("Type 'done' when you are finished connecting nodes.")
        while True:
            parent_input = input("\nEnter parent node name (or 'done' to finish): ").strip()
            if parent_input.lower() == 'done':
                break
            children_input = input(f"Enter children names for parent '{parent_input}' (separated by commas): ")

            parent_node = get_or_create(parent_input)
            child_names = [c.strip() for c in children_input.split(",")]

            parent_node.children = [get_or_create(c_name) for c_name in child_names if c_name]

        root_name = input("\nEnter the name of the root node to start search: ").strip()
        if root_name not in nodes:
            print("Error: Root node not found!")
            return None
        return nodes[root_name]
    except Exception as e:
        print(f"An error occurred with your inputs: {e}")
        return None

# To run with custom inputs, call: root = build_tree_from_input()
# For now, let's provide a quick interactive prompt interface
root = build_tree_from_input()

if root:
    print("\nTree Structure:")
    print_tree(root)

    # Run search
    pruned_branches.clear()
    optimal_val, optimal_path = alpha_beta(root, 0, float('-inf'), float('inf'), True, [])

    print("\n--- Results ---")
    print(f"Value of Root Node: {optimal_val}")
    print(f"Optimal Path: {' -> '.join(optimal_path)}")
    print(f"Pruned Paths/Branches: {pruned_branches if pruned_branches else 'None'}")
else:
    print("Could not construct the tree. Please try executing the cell again with valid inputs.")

--- Build Your Custom Tree ---
We will construct a tree where internal nodes branch, and leaf nodes hold numeric values.

Enter the number of leaf nodes: 8
Enter leaf node name: A
Enter evaluation value for leaf 'A': 10
Enter leaf node name: B
Enter evaluation value for leaf 'B': 9
Enter leaf node name: C
Enter evaluation value for leaf 'C': 14
Enter leaf node name: D
Enter evaluation value for leaf 'D': 18
Enter leaf node name: E
Enter evaluation value for leaf 'E': 5
Enter leaf node name: F
Enter evaluation value for leaf 'F': 4
Enter leaf node name: G
Enter evaluation value for leaf 'G': 50
Enter leaf node name: H
Enter evaluation value for leaf 'H': 3

Now define internal connections. Enter parent and its children (comma-separated).
Type 'done' when you are finished connecting nodes.

Enter parent node name (or 'done' to finish): P
Enter children names for parent 'P' (separated by commas): A,B

Enter parent node name (or 'done' to finish): Q
Enter children names for parent 'Q' (sep

In [ ]:
# Constructing the exact tree from the screenshot
# Level 4: Leaf Nodes
L1 = Node('10', value=10)
L2 = Node('9', value=9)
L3 = Node('14', value=14)
L4 = Node('18', value=18)
L5 = Node('5', value=5)
L6 = Node('4', value=4)
L7 = Node('50', value=50)
L8 = Node('3', value=3)

# Level 3: MAX Nodes
D = Node('D (MAX)', children=[L1, L2])
E = Node('E (MAX)', children=[L3, L4])
F = Node('F (MAX)', children=[L5, L6])
G = Node('G (MAX)', children=[L7, L8])

# Level 2: MIN Nodes
B = Node('B (MIN)', children=[D, E])
C = Node('C (MIN)', children=[F, G])

# Level 1: Root Node (MAX)
root_screenshot = Node('A (MAX)', children=[B, C])

print("Tree Structure:")
print_tree(root_screenshot)

# Reset and run search
pruned_branches.clear()
optimal_val, optimal_path = alpha_beta(root_screenshot, 0, float('-inf'), float('inf'), True, [])

print("\n--- Alpha-Beta Pruning Results ---")
print(f"Value of Root Node: {optimal_val}")
print(f"Optimal Path taken: {' -> '.join(optimal_path)}")
print(f"Pruned Paths/Branches: {pruned_branches}")

Tree Structure:
A (MAX)
  B (MIN)
    D (MAX)
      10 (val=10)
      9 (val=9)
    E (MAX)
      14 (val=14)
      18 (val=18)
  C (MIN)
    F (MAX)
      5 (val=5)
      4 (val=4)
    G (MAX)
      50 (val=50)
      3 (val=3)

--- Alpha-Beta Pruning Results ---
Value of Root Node: 10
Optimal Path taken: A (MAX) -> B (MIN) -> D (MAX) -> 10
Pruned Paths/Branches: [('E (MAX)', '18'), ('C (MIN)', 'G (MAX)')]
